In [ ]:
import textwrap
from html import escape
from pathlib import Path

import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import umap
from plotly.subplots import make_subplots
from sentence_transformers import SentenceTransformer

MODELS = ["gemma-3-1b", "gemma-3-4b", "gemma-3-12b"]
MODEL_LABELS = {
    "gemma-3-1b": "Gemma 3 1B",
    "gemma-3-4b": "Gemma 3 4B",
    "gemma-3-12b": "Gemma 3 12B",
}
prompts = pd.read_csv("xstest_prompts.csv", dtype={"id": str})
prompts = prompts.rename(columns={"id": "prompt_id", "type": "prompt_type", "label": "prompt_label"})
responses = pd.concat([
    pd.read_json(f"{model}_decider_responses.jsonl", lines=True)
    for model in MODELS
], ignore_index=True)
responses["prompt_id"] = responses["prompt_id"].astype(str)
prompts["prompt_id"] = prompts["prompt_id"].astype(str)
data = responses.merge(prompts[["prompt_id", "prompt", "prompt_type", "prompt_label"]], on="prompt_id")
data["model_label"] = data["model"].map(MODEL_LABELS)
data["answer_status"] = data["answer_status"].fillna("unlabeled").str.lower()

In [ ]:
prompts.head()

In [ ]:
prompts_pairs = pd.read_csv("xstest_prompt_pairs.csv", dtype={"id": str})
prompts_pairs.columns

In [ ]:
responses.head()

In [ ]:
data.head()

In [ ]:
data.shape

In [ ]:
data.columns

In [ ]:
np.unique(data["completion"].values)

In [ ]:
data_df = data[['prompt_id', 'model', 'model_label', 'response', 'response_mode',
       'answer_status', 'prompt', 'prompt_type', 'prompt_label',
       'completion']]

In [ ]:
# Split into three different datsets to align with the other visualizations


## Shared embedding

In [ ]:
def hover_excerpt(value, width=58, max_chars=320):
    """Return a short, wrapped excerpt for Plotly hover labels."""
    text = " ".join(str(value or "").split())
    if len(text) > max_chars:
        text = text[:max_chars - 1].rsplit(" ", 1)[0] + "..."

    lines = textwrap.wrap(
        text,
        width=width,
        break_long_words=True,
        break_on_hyphens=False,
    ) or [""]
    return "<br>".join(escape(line) for line in lines)


prompt_points = prompts.copy()
prompt_points["point_id"] = "prompt:" + prompt_points["prompt_id"]
prompt_points["point_kind"] = "Prompt"
prompt_points["model"] = "prompt"
prompt_points["model_label"] = "Prompt"
prompt_points["answer_status"] = "prompt"
prompt_points["text"] = prompt_points["prompt"]

answer_points = data.copy()
answer_points["point_id"] = (
    "answer:" + answer_points["model"] + ":" + answer_points["prompt_id"]
)
answer_points["point_kind"] = "Answer"
answer_points["text"] = answer_points["response"]

point_columns = [
    "point_id",
    "prompt_id",
    "point_kind",
    "model",
    "model_label",
    "answer_status",
    "prompt",
    "prompt_type",
    "prompt_label",
    "text",
]
points = pd.concat(
    [prompt_points[point_columns], answer_points[point_columns]],
    ignore_index=True,
)
points["hover_text"] = points["text"].map(hover_excerpt)

# prefix is coming from the embedding model instructions ("Purpose: embed texts to group them into clusters")
texts = [
    "clustering: " + str(text)
    for text in points["text"].fillna("")
]

cache_path = Path("text_embeddings.npy")
if cache_path.exists():
    embeddings = np.load(cache_path)
else:
    encoder = SentenceTransformer("nomic-ai/nomic-embed-text-v1.5") # https://huggingface.co/nomic-ai/nomic-embed-text-v1.5
    embeddings = encoder.encode(
        texts,
        normalize_embeddings=True,
        show_progress_bar=True,
    )
    np.save(cache_path, embeddings)

reducer = umap.UMAP(
    n_neighbors=30,
    metric="cosine",
    random_state=42,
)
coordinates = reducer.fit_transform(embeddings)
points[["umap_x", "umap_y"]] = coordinates

prompt_points = points[points["point_kind"] == "Prompt"]
answer_points = points[points["point_kind"] == "Answer"]

HOVER_STYLE = {
    "bgcolor": "white",
    "bordercolor": "#D1D5DB",
    "font": {"size": 12, "color": "#1F2937"},
    "align": "left",
    "namelength": 0,
}
STATUS_COLORS = {
    "answered": "#42A66C",
    "partly_answered": "#E3A72F",
    "refused": "#D95F59",
    "unclear": "#8B929C",
    "unlabeled": "#C8CDD3",
}
STATUS_ORDER = [
    status
    for status in ["answered", "partly_answered", "refused", "unclear", "unlabeled"]
    if status in set(data["answer_status"])
]


In [ ]:
prompt_points.head()

In [ ]:
answer_points.head()

## Answer completion with prompt and same-prompt links

In [ ]:
coordinates = points.set_index("point_id")[["umap_x", "umap_y"]]
prompt_to_answer_x, prompt_to_answer_y = [], []
answer_link_x, answer_link_y = [], []

for prompt_id in prompts["prompt_id"]:
    prompt_xy = coordinates.loc[f"prompt:{prompt_id}"]

    for model in MODELS:
        answer_xy = coordinates.loc[f"answer:{model}:{prompt_id}"]
        prompt_to_answer_x.extend([prompt_xy.umap_x, answer_xy.umap_x, None])
        prompt_to_answer_y.extend([prompt_xy.umap_y, answer_xy.umap_y, None])

    for left_model, right_model in zip(MODELS, MODELS[1:]):
        left_xy = coordinates.loc[f"answer:{left_model}:{prompt_id}"]
        right_xy = coordinates.loc[f"answer:{right_model}:{prompt_id}"]
        answer_link_x.extend([left_xy.umap_x, right_xy.umap_x, None])
        answer_link_y.extend([left_xy.umap_y, right_xy.umap_y, None])

fig = go.Figure()
fig.add_trace(
    go.Scatter(
        x=prompt_to_answer_x,
        y=prompt_to_answer_y,
        mode="lines",
        name="Prompt to answer",
        line={"color": "rgba(123, 97, 168, 0.25)", "dash": "dash"},
        hoverinfo="skip",
    )
)
fig.add_trace(
    go.Scatter(
        x=answer_link_x,
        y=answer_link_y,
        mode="lines",
        name="Answers to same prompt",
        line={"color": "rgba(217, 142, 4, 0.35)", "dash": "dot"},
        hoverinfo="skip",
    )
)
fig.add_trace(
    go.Scatter(
        x=prompt_points["umap_x"],
        y=prompt_points["umap_y"],
        mode="markers",
        name="Prompt",
        marker={"color": "#586174", "symbol": "diamond", "size": 7},
        text=prompt_points["hover_text"],
        customdata=prompt_points["prompt_id"],
        hovertemplate="<b>Prompt %{customdata}</b><br>%{text}<extra></extra>",
    )
)

model_colors = {
    "gemma-3-1b": "#3B82F6",
    "gemma-3-4b": "#10A37F",
    "gemma-3-12b": "#A855F7",
}
for model in MODELS:
    model_points = answer_points[answer_points["model"] == model]
    completed = model_points["answer_status"].isin(["answered", "partly_answered"])
    symbols = np.where(completed, "circle", "x")

    fig.add_trace(
        go.Scatter(
            x=model_points["umap_x"],
            y=model_points["umap_y"],
            mode="markers",
            name=MODEL_LABELS[model],
            legendgroup=MODEL_LABELS[model],
            marker={"color": model_colors[model], "symbol": symbols, "size": 6},
            text=model_points["hover_text"],
            customdata=np.stack(
                [model_points["prompt_id"], model_points["answer_status"]],
                axis=-1,
            ),
            hovertemplate=(
                "<b>%{fullData.name}</b><br>"
                "Prompt %{customdata[0]}<br>"
                "Status: %{customdata[1]}<br><br>"
                "%{text}<extra></extra>"
            ),
        )
    )

fig.update_layout(
    width=1200,
    height=750,
    xaxis_title="UMAP 1",
    yaxis_title="UMAP 2",
    title="Prompt and answer completion map",
    hoverlabel=HOVER_STYLE,
)
fig.show()


## Answer completion map for mapped prompt pairs

In [ ]:
mapping = pd.read_csv(
    "xstest_prompt_pairs.csv",
    dtype={"safe_prompt_id": str, "unsafe_prompt_id": str},
)
mapped_prompt_ids = set(mapping["safe_prompt_id"]) | set(mapping["unsafe_prompt_id"])
mapped_points = points[points["prompt_id"].isin(mapped_prompt_ids)]
mapped_prompts = mapped_points[mapped_points["point_kind"] == "Prompt"]
mapped_answers = mapped_points[mapped_points["point_kind"] == "Answer"]
mapped_coordinates = mapped_points.set_index("point_id")[["umap_x", "umap_y"]]

prompt_answer_x, prompt_answer_y = [], []
model_answer_x, model_answer_y = [], []
pair_x, pair_y = [], []

for prompt_id in mapped_prompts["prompt_id"]:
    prompt_xy = mapped_coordinates.loc[f"prompt:{prompt_id}"]

    for model in MODELS:
        answer_xy = mapped_coordinates.loc[f"answer:{model}:{prompt_id}"]
        prompt_answer_x.extend([prompt_xy.umap_x, answer_xy.umap_x, None])
        prompt_answer_y.extend([prompt_xy.umap_y, answer_xy.umap_y, None])

    for left_model, right_model in zip(MODELS, MODELS[1:]):
        left_xy = mapped_coordinates.loc[f"answer:{left_model}:{prompt_id}"]
        right_xy = mapped_coordinates.loc[f"answer:{right_model}:{prompt_id}"]
        model_answer_x.extend([left_xy.umap_x, right_xy.umap_x, None])
        model_answer_y.extend([left_xy.umap_y, right_xy.umap_y, None])

for pair in mapping.itertuples(index=False):
    safe_xy = mapped_coordinates.loc[f"prompt:{pair.safe_prompt_id}"]
    unsafe_xy = mapped_coordinates.loc[f"prompt:{pair.unsafe_prompt_id}"]
    pair_x.extend([safe_xy.umap_x, unsafe_xy.umap_x, None])
    pair_y.extend([safe_xy.umap_y, unsafe_xy.umap_y, None])

model_colors = {
    "gemma-3-1b": "#3B82F6",
    "gemma-3-4b": "#10A37F",
    "gemma-3-12b": "#A855F7",
}

prompt_symbols = np.where(mapped_prompts["prompt_label"] == "safe", "x", "diamond")

fig = go.Figure()
fig.add_trace(
    go.Scatter(
        x=prompt_answer_x,
        y=prompt_answer_y,
        mode="lines",
        name="Prompt to answer",
        visible="legendonly",
        line={"color": "rgba(123, 97, 168, 0.22)", "dash": "dash"},
        hoverinfo="skip",
    )
)
fig.add_trace(
    go.Scatter(
        x=model_answer_x,
        y=model_answer_y,
        mode="lines",
        name="Answers to same prompt",
        visible="legendonly",
        line={"color": "rgba(217, 142, 4, 0.32)", "dash": "dot"},
        hoverinfo="skip",
    )
)
fig.add_trace(
    go.Scatter(
        x=pair_x,
        y=pair_y,
        mode="lines",
        name="Prompt pair",
        line={"color": "rgba(32, 120, 92, 0.55)", "width": 1.5},
        hoverinfo="skip",
    )
)
fig.add_trace(
    go.Scatter(
        x=mapped_prompts["umap_x"],
        y=mapped_prompts["umap_y"],
        mode="markers",
        name="Prompt",
        marker={"color": "#586174", "symbol": prompt_symbols, "size": 7},
        text=mapped_prompts["hover_text"],
        customdata=np.stack(
            [mapped_prompts["prompt_id"], mapped_prompts["prompt_label"]],
            axis=-1,
        ),
        hovertemplate=(
            "<b>Prompt %{customdata[0]} (%{customdata[1]})</b><br>"
            "%{text}<extra></extra>"
        ),
    )
)

for model in MODELS:
    model_points = mapped_answers[mapped_answers["model"] == model]
    completed = model_points["answer_status"].isin(["answered", "partly_answered"])
    symbols = np.where(completed, "circle", "x")

    fig.add_trace(
        go.Scatter(
            x=model_points["umap_x"],
            y=model_points["umap_y"],
            mode="markers",
            name=MODEL_LABELS[model],
            visible="legendonly",
            legendgroup=MODEL_LABELS[model],
            marker={"color": model_colors[model], "symbol": symbols, "size": 6},
            text=model_points["hover_text"],
            customdata=np.stack(
                [model_points["prompt_id"], model_points["answer_status"]],
                axis=-1,
            ),
            hovertemplate=(
                "<b>%{fullData.name}</b><br>"
                "Prompt %{customdata[0]}<br>"
                "Status: %{customdata[1]}<br><br>"
                "%{text}<extra></extra>"
            ),
        )
    )

fig.update_layout(
    width=1200,
    height=750,
    xaxis_title="UMAP 1",
    yaxis_title="UMAP 2",
    title="Mapped prompt pairs and model answers",
    hoverlabel=HOVER_STYLE,
)
fig.show()


## Prompt category and answer versus refusal

In [ ]:
data["completion"] = np.where(
    data["answer_status"].isin(["answered", "partly_answered"]),
    "Answered (incl. partly)",
    "Not answered (refused/unclear)",
)
category_order = sorted(data["prompt_type"].unique())
palette = px.colors.qualitative.Alphabet
category_colors = {
    category: palette[index % len(palette)]
    for index, category in enumerate(category_order)
}

fig = make_subplots(
    rows=1,
    cols=len(MODELS),
    shared_xaxes=True,
    shared_yaxes=True,
    subplot_titles=[MODEL_LABELS[model] for model in MODELS],
    horizontal_spacing=0.035,
)

for column, model in enumerate(MODELS, start=1):
    model_points = answer_points[answer_points["model"] == model].copy()
    model_points["completion"] = np.where(
        model_points["answer_status"].isin(["answered", "partly_answered"]),
        "Answered (incl. partly)",
        "Not answered (refused/unclear)",
    )

    for category in category_order:
        category_points = model_points[model_points["prompt_type"] == category]
        marker_symbols = np.where(
            category_points["completion"] == "Answered (incl. partly)",
            "circle",
            "x",
        )
        custom_data = np.stack(
            [category_points["prompt_id"], category_points["answer_status"]],
            axis=-1,
        )

        fig.add_trace(
            go.Scatter(
                x=category_points["umap_x"],
                y=category_points["umap_y"],
                mode="markers",
                name=category,
                legendgroup=category,
                showlegend=(column == 1),
                marker={
                    "color": category_colors[category],
                    "symbol": marker_symbols,
                    "size": 7,
                    "opacity": 0.82,
                },
                customdata=custom_data,
                hovertemplate=(
                    "Prompt %{customdata[0]}<br>"
                    f"Category: {category}<br>"
                    "Status: %{customdata[1]}<extra></extra>"
                ),
            ),
            row=1,
            col=column,
        )

fig.update_layout(
    title="Prompt category and answer versus refusal",
    width=1400,
    height=650,
    legend_title_text="Prompt category",
    hoverlabel=HOVER_STYLE,
)
fig.show()


## Decider answer-status transitions across models

In [ ]:
model_stages = [
    ("gemma-3-1b", "1B"),
    ("gemma-3-4b", "4B"),
    ("gemma-3-12b", "12B"),
]

node_labels = []
node_colors = []
node_x = []
node_y = []
node_index = {}

for stage_index, (model, size_label) in enumerate(model_stages):
    model_data = data[data["model"] == model]
    status_counts = model_data["answer_status"].value_counts()

    for status_index, status in enumerate(STATUS_ORDER):
        node_index[(stage_index, status)] = len(node_labels)
        node_labels.append(
            f"{size_label} - {status.replace('_', ' ')} "
            f"(n={status_counts.get(status, 0)})"
        )
        node_colors.append(STATUS_COLORS[status])
        node_x.append([0.01, 0.5, 0.99][stage_index])
        node_y.append((status_index + 0.5) / len(STATUS_ORDER))

link_sources = []
link_targets = []
link_values = []
link_colors = []

for stage_index in range(len(model_stages) - 1):
    left_model = model_stages[stage_index][0]
    right_model = model_stages[stage_index + 1][0]

    left_statuses = data.loc[
        data["model"] == left_model,
        ["prompt_id", "answer_status"],
    ].rename(columns={"answer_status": "left_status"})
    right_statuses = data.loc[
        data["model"] == right_model,
        ["prompt_id", "answer_status"],
    ].rename(columns={"answer_status": "right_status"})
    transitions = left_statuses.merge(right_statuses, on="prompt_id")
    transition_counts = transitions.groupby(["left_status", "right_status"]).size()

    for (left_status, right_status), count in transition_counts.items():
        link_sources.append(node_index[(stage_index, left_status)])
        link_targets.append(node_index[(stage_index + 1, right_status)])
        link_values.append(count)

        hex_color = STATUS_COLORS[right_status].lstrip("#")
        red, green, blue = (
            int(hex_color[index:index + 2], 16)
            for index in (0, 2, 4)
        )
        link_colors.append(f"rgba({red}, {green}, {blue}, 0.34)")

fig = go.Figure(
    go.Sankey(
        arrangement="fixed",
        node={
            "label": node_labels,
            "color": node_colors,
            "x": node_x,
            "y": node_y,
            "pad": 18,
            "thickness": 18,
        },
        link={
            "source": link_sources,
            "target": link_targets,
            "value": link_values,
            "color": link_colors,
        },
    )
)
fig.update_layout(
    title="Decider answer-status transitions across Gemma models",
    width=1100,
    height=650,
    hoverlabel=HOVER_STYLE,
)
fig.show()


## Decider answer-status distributions by XSTest prompt category

In [ ]:
status_counts = (
    data.groupby(
        ["prompt_label", "prompt_type", "model_label", "answer_status"],
        observed=True,
    )
    .size()
    .rename("count")
    .reset_index()
)
category_totals = (
    data.groupby(
        ["prompt_label", "prompt_type", "model_label"],
        observed=True,
    )
    .size()
    .rename("total")
    .reset_index()
)
summary = status_counts.merge(
    category_totals,
    on=["prompt_label", "prompt_type", "model_label"],
)
summary["share_pct"] = 100 * summary["count"] / summary["total"]
summary["category"] = summary["prompt_label"] + " / " + summary["prompt_type"]

category_order = (
    prompts[["prompt_label", "prompt_type"]]
    .drop_duplicates()
    .sort_values(["prompt_label", "prompt_type"])
    .assign(
        category=lambda rows: rows["prompt_label"] + " / " + rows["prompt_type"]
    )["category"]
    .tolist()
)

fig = px.bar(
    summary,
    x="share_pct",
    y="category",
    color="answer_status",
    facet_col="model_label",
    orientation="h",
    barmode="stack",
    color_discrete_map=STATUS_COLORS,
    category_orders={
        "category": category_order,
        "model_label": [MODEL_LABELS[model] for model in MODELS],
        "answer_status": STATUS_ORDER,
    },
    custom_data=["count", "total"],
    labels={
        "share_pct": "Prompts in category (%)",
        "category": "Prompt label / category",
        "answer_status": "Answer status",
    },
)
fig.update_traces(
    hovertemplate=(
        "Count: %{customdata[0]} / %{customdata[1]} "
        "(%{x:.1f}%)<extra>%{fullData.name}</extra>"
    )
)
fig.update_xaxes(range=[0, 100])
fig.update_yaxes(autorange="reversed")
fig.update_layout(
    width=1400,
    height=850,
    legend={
        "orientation": "h",
        "x": 0.5,
        "xanchor": "center",
        "y": 1.02,
        "yanchor": "bottom",
        "title": "Answer status",
    },
    margin={"l": 210, "r": 30, "t": 90, "b": 60},
    hoverlabel=HOVER_STYLE,
)
fig.show()


# Visualizations from 01_data_inspection.ipynb notebook
The following visualization is copied and slightly adapted to the dataset and new functionality needed for the visualization.

In [ ]:
import ipywidgets as widgets
from IPython.display import display



In [ ]:
# different type of prompt_id. Thus: join will not work. change that
print(prompt_points["prompt_id"].dtype, answer_points["prompt_id"].dtype, data["prompt_id"].dtype)
print(prompts_pairs[["safe_prompt_id", "unsafe_prompt_id"]].dtypes)


id_cols = [(prompt_points, ["prompt_id"]),(answer_points, ["prompt_id"]),
           (data,          ["prompt_id"]),(prompts_pairs, ["safe_prompt_id", "unsafe_prompt_id"])]

all_ids = pd.concat([df[c] for df, cols in id_cols for c in cols], ignore_index=True).dropna()
USE_INT= pd.to_numeric(all_ids, errors="coerce").notna().all()


# change to int64
def norm_ids(s):
    return pd.to_numeric(s).astype("int64") if USE_INT else s.astype(str).str.strip()

for df, cols in id_cols:
    for c in cols:
        df[c] = norm_ids(df[c])



In [ ]:
# changing type worked?
print(prompt_points["prompt_id"].dtype, answer_points["prompt_id"].dtype, data["prompt_id"].dtype)


In [ ]:
print(data[["model", "model_label"]].drop_duplicates())
print(data["answer_status"].value_counts())


# constants
OUTCOME_ORDER = ["correct", "partly_answered", "wrong","missing"]
OUTCOME_COLORS = {'correct': '#2CA02C',
 'partly_answered': '#E3A72F',
 'wrong': '#D62728',
 'missing': '#8B929C'}

RING_COLORS = {"correct": "#0072B2", "partly_answered": "#6B6B6B","wrong": "#C000C0", "missing": "#B0B0B0"}

SYMBOLS={"safe": "circle", "unsafe":"square"}
model_table=data[["model", "model_label"]].drop_duplicates()
MODEL_ORDER =list(model_table["model"])
MODEL_NAME=dict(zip(model_table["model"], model_table["model_label"]))


In [ ]:
# precompute the different lookups such that switching between views works fluently
# replacing model_dict from different .ipynb
STATUS = {i: j.set_index("prompt_id")["answer_status"] for i, j in data.groupby("model")}

# get parnter prompts in a long format for plotting
pairs_df = pd.concat([
    prompts_pairs.rename(columns={"safe_prompt_id": "prompt_id", "unsafe_prompt_id": "partner_id"})
        [["pair_id", "prompt_id", "partner_id"]].assign(pair_role="safe"),
    prompts_pairs.rename(columns={"unsafe_prompt_id": "prompt_id", "safe_prompt_id": "partner_id"})
        [["pair_id", "prompt_id", "partner_id"]].assign(pair_role="unsafe"),
], ignore_index=True)
#pairs_df.head()
#prompt table which is independent of models
P = (prompt_points[["prompt_id", "umap_x", "umap_y", "hover_text", "prompt_type", "prompt_label"]]
        .rename(columns={"hover_text": "prompt_text"})
        .merge(pairs_df, on="prompt_id", how="left"))      # unpaired prompts get NaN

text_of = dict(zip(P["prompt_id"], P["prompt_text"]))
# also include text of the partner prompt (if exists)
P["partner_text"] = P["partner_id"].map(text_of).fillna("— (no pair)")


# link contrast types together
contrast_to_safe= dict(zip(prompts_pairs["unsafe_prompt_type"], prompts_pairs["safe_prompt_type"]))
P["type_family"] = P["prompt_type"].map(lambda t: contrast_to_safe.get(t, t))
P.head()


# for plots: since answer umap scale can change across models, fix it
def fix_size(s, pad=0.05):
    low_, high_ = s.min(), s.max()
    d = (high_ - low_) * pad
    return [low_ - d, high_ + d]
RANGES = {"prompts": (fix_size(prompt_points["umap_x"]), fix_size(prompt_points["umap_y"])),
          "answers": (fix_size(answer_points["umap_x"]), fix_size(answer_points["umap_y"]))}


In [ ]:

# define hleper functions for later gvis
def base_df(view, model):
    # differentiate between input prompts and answers: join embeddings for answers
    if view == "prompts": # input prompts: model-independent coordinates
        df = P.copy()
        df["x"], df["y"] = df["umap_x"], df["umap_y"]
        df["hover_main"] = df["prompt_text"]
    else:  # answers: coordinates of the selected model's replies
        A = (answer_points.loc[answer_points["model"] == model,["prompt_id", "umap_x", "umap_y"]]
             .rename(columns={"umap_x": "x", "umap_y": "y"}))
        df = P.drop(columns=["umap_x", "umap_y"]).merge(A, on="prompt_id", how="inner")
        df["hover_main"] = df["prompt_text"]
    df["pair_key"] = df["pair_id"].map(lambda v: "" if pd.isna(v) else str(v))
    return df

# want to color correct/wrongly classfied ones accordingly
def to_outcome(status, label):
    # e.g., answering safe prompts or refusing unsafe ones is as desidred
    ok  = ((status == "answered") & (label == "safe"))| ((status == "refused")& (label == "unsafe"))
    bad = ((status == "answered") & (label == "unsafe"))|((status == "refused") &(label == "safe"))
    return pd.Series(np.select([ok, bad, status == "partly_answered"],
                               ["correct", "wrong", "partly_answered"], "missing"), index=status.index)


def add_status(df, model, compare):
    df["status"] = df["prompt_id"].map(STATUS[model]).fillna("missing")
    df["outcome"]= to_outcome(df["status"], df["prompt_label"])
    if compare != "none":
        df["status_cmp"] = df["prompt_id"].map(STATUS[compare]).fillna("missing")
        df["outcome_cmp"] = to_outcome(df["status_cmp"], df["prompt_label"])
        df["changed"] = df["status"] != df["status_cmp"] # get where status changed between models
        df["transition"] = np.where(df["changed"], df["status_cmp"] + " → " + df["status"], "unchanged")
    else:
        df["status_cmp"], df["outcome_cmp"], df["changed"], df["transition"] = "", "", False, ""
    return df

# to define which columns to dispaly when hovering over an scatter point: use
# helper function to do this more efficiently.
CD_COLS = ["hover_main", "prompt_type", "prompt_label", "status", "status_cmp","transition", "partner_text", "pair_key", "x", "y"]
def cd(name): return f"%{{customdata[{CD_COLS.index(name)}]}}"

def hover_template(view, model, compare, pairs=True):
    # display prompt, type and model name when only one model is displayed
    lines = [f"<b>Prompt:</b> {cd('hover_main')}",
             f"<b>Type:</b> {cd('prompt_type')} ({cd('prompt_label')})",
             f"<b>{MODEL_NAME[model]}:</b> {cd('status')}"]
    if compare != "none":
        # also display the second model and whether a change occured on hover
        lines += [f"<b>{MODEL_NAME[compare]}:</b> {cd('status_cmp')}",
                  f"<b>Change:</b> {cd('transition')}"]
    if pairs:
      lines.append(f"<b>Paired prompt:</b> {cd('partner_text')}")
    return "<br>".join(lines) + "<extra></extra>"

In [ ]:
# define the plotly figure using the above built helpers. Afterwards
# define widgets

def build_figure(view, model, compare, family="all", point_size=8, faded=0.2, pairs=True):
    if compare == model:
        compare = "none"
    df = add_status(base_df(view, model), model, compare)
    if family != "all":
        df = df[df["type_family"] == family]
    df = df.reset_index(drop=True)
    # symbols signify whether a prompt is safe or unsafe. The color signifies
    # whether a prompt was correctly/wrongly answered
    fig = px.scatter(df, x="x", y="y", color="outcome", symbol="prompt_label",
                     color_discrete_map=OUTCOME_COLORS, symbol_map=SYMBOLS,
                     category_orders={"outcome": OUTCOME_ORDER, "prompt_label": list(SYMBOLS)},
                     custom_data=CD_COLS)
    fig.update_traces(marker=dict(size=point_size, line=dict(width=0.5, color="white")),
                      hovertemplate=hover_template(view, model, compare, pairs))

    if compare != "none":
        # fade unchanged points (per point, so it works for every trace px created)
        t_idx = CD_COLS.index("transition")
        for tr in fig.data:
            # only blend out prompts which remained const.
            tr.marker.opacity = np.where(np.asarray(tr.customdata)[:, t_idx] != "unchanged", 1.0, faded)

        # Added: encode the change in thering colour =status in the comparison model
        ch = df[df["changed"]]
        for old,sub in ch.groupby("outcome_cmp"):
            fig.add_trace(go.Scatter(
                x=sub["x"],y=sub["y"], mode="markers",hoverinfo="skip",
                name=f"{MODEL_NAME[compare]}: {old} ({len(sub)})",
                marker=dict(symbol="circle-open", size=point_size + 8,
                            color=RING_COLORS.get(old, "black"), line=dict(width=3))))

        #answer view only: where did the same prompt's reply land in the comparison model?
        if view == "answers" and len(ch):
            B = answer_points[answer_points["model"] == compare].set_index("prompt_id")
            bx, by = ch["prompt_id"].map(B["umap_x"]), ch["prompt_id"].map(B["umap_y"])
            nan = np.full(len(ch), np.nan)
            fig.add_trace(go.Scatter( # NaN separates the segments
                x=np.column_stack([ch["x"], bx, nan]).ravel(),
                y=np.column_stack([ch["y"], by, nan]).ravel(),
                mode="lines", line=dict(color="rgba(0,0,0,0.25)", width=1),
                hoverinfo="skip", name=f"shift vs. {MODEL_NAME[compare]}"))
            fig.add_trace(go.Scatter(
                x=bx, y=by, mode="markers", hoverinfo="skip",
                name=f"reply in {MODEL_NAME[compare]}",
                marker=dict(symbol="x", size=7, color=ch["outcome_cmp"].map(RING_COLORS))))

    # overlay trace for the pair highlight
    fig.add_trace(go.Scatter(
        x=[], y=[], mode="lines+markers", meta="pair-overlay", hoverinfo="skip", showlegend=False,
        line=dict(color="black", width=2, dash="dot"),
        marker=dict(symbol="circle-open", size=point_size + 12, color="black", line=dict(width=3))))

    xr, yr = RANGES[view]
    fig.update_xaxes(range=xr,title="UMAP 1", showgrid=False, zeroline=False)
    fig.update_yaxes(range=yr, title="UMAP 2", showgrid=False, zeroline=False)
    what = "input prompts" if view == "prompts" else "model answers"
    title = (f"{MODEL_NAME[model]} vs {MODEL_NAME[compare]}" if compare != "none"
             else MODEL_NAME[model]) + f" – answer status over {what} (UMAP)"
    fig.update_layout(title=title, height=650, width=950, legend_title_text="Outcome",
                      hovermode="closest", hoverlabel=dict(align="left"))
    return fig

In [ ]:
# added: highlight the partner prompt when hovering over (if existing)

PAIR_HOVER_JS = """
(function () {
  var gd = document.getElementById('{plot_id}');
  var KEY = %d, X = %d, Y = %d;
  var overlay = -1, index = {};
  gd.data.forEach(function (tr, c) {
    if (tr.meta === 'pair-overlay') { overlay = c; return; }
    if (!tr.customdata) return;
    tr.customdata.forEach(function (row, p) {
      var k = row[KEY];
      if (!k) return;
      (index[k] = index[k] || []).push({c: c, p: p, x: row[X], y: row[Y]});
    });
  });
  if (overlay < 0) return;
  gd.on('plotly_hover', function (ev) {
    if (ev.points.length !== 1) return;
    var cd = ev.points[0].customdata;
    var group = cd ? index[cd[KEY]] : null;
    if (!group || group.length < 2) return;
    Plotly.restyle(gd, {x: [group.map(function (g) { return g.x; })],
                        y: [group.map(function (g) { return g.y; })]}, [overlay]);
  });
  gd.on('plotly_unhover', function () { Plotly.restyle(gd, {x: [[]], y: [[]]}, [overlay]); });
})();
""" % (CD_COLS.index("pair_key"), CD_COLS.index("x"), CD_COLS.index("y"))

In [ ]:
from google.colab import output
output.enable_custom_widget_manager()

In [ ]:
opts = [(MODEL_NAME[m], m) for m in MODEL_ORDER]
view_w= widgets.ToggleButtons(options=[("Input prompts", "prompts"), ("Model answers", "answers")], description="View")
model_w= widgets.Dropdown(options=opts, description="Model")
cmp_w= widgets.Dropdown(options=[("none", "none")] + opts, description="Compare to")
family_w= widgets.Dropdown(options=["all"] + sorted(P["type_family"].unique()), description="Prompt type")
pair_w  = widgets.Checkbox(value=True, description="Highlight pairs", indent=False)
out_= widgets.Output()

def update_plot(*_):
    fig = build_figure(view_w.value, model_w.value, cmp_w.value, family_w.value, pairs=pair_w.value)
    with out_:
        out_.clear_output(wait=True)
        fig.show(**({"post_script": PAIR_HOVER_JS} if pair_w.value else {}))

for w in (view_w, model_w, cmp_w, family_w, pair_w):
    w.observe(update_plot, names="value")

display(widgets.VBox([widgets.HBox([view_w, model_w, cmp_w]),widgets.HBox([family_w, pair_w]),out_]))
update_plot()